# Lab 1: The Data Portfolio — The Economic Lens
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | Part 0 25 min + Parts 1-3 34 min

---

**Format:** This lab contains **deliberately flawed code and analysis**. Your job:
1. Run the code
2. Identify what is wrong (not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Measure what the flaw costs — the size and direction of the bias it introduces

**Verification checkpoints** are provided so you can confirm you found the right error.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 1, the fix cell.** It starts empty. You write the corrected valuation (about three lines) and print the new top five.
> - **Part 2, two blanks** in the cell that measures the bias, one line each.
> - **Part 3, five blanks** in `profile_dataframe()`, one line each.
> - **Text answers:** the diagnosis questions in Parts 1 and 2, the three economics questions after the Part 1 chart, and the numbers you copy into the README prompt at the end.
>
> **You run and read:** everything else. Part 0 is working code to learn from, not to retype, and the check cells after Parts 2 and 3 tell you whether your blanks are right.
>
> Writing code from a blank cell is what Problem Set 1 grades. Its ungraded warm-up, linked from the PS1 assignment, is the practice. If you are reading this lab after its due date, read it next to the Lab 1 solutions as a worked reference for PS1.

## Setup

The data is The Economist's Big Mac index, read straight from their live
repository. They revise it twice a year, so a later run can gain a period or
move the last decimal of a number quoted in this lab.

In [ ]:
# GIVEN — run as-is
# Setup: libraries and the Big Mac data
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/TheEconomist/big-mac-data/master/output-data/big-mac-full-index.csv"
df = pd.read_csv(url, parse_dates=["date"])   # read the date column as dates, not text
print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Countries: {df['name'].nunique()}, Periods: {df['date'].nunique()}")

<!-- 5200-onramp -->
## Part 0: Build It First (GUIDED — 25 min)

**Read this before anything else.** The rest of this lab hands you code that is
deliberately wrong and asks you to find the error. You cannot spot a wrong
answer without having seen the right one, so we build the correct version first.

This course assumes no prior programming. Part 0 of Labs 1 to 5 teaches the
slice of Python each lab needs, here in the lab, next to the data. There is no
separate primer to find.

**This lab's slice:** the notebook itself, f-strings, the DataFrame, the
boolean mask, lists and `for`, `.groupby()`, dictionaries, and `def`. That is
everything Parts 1 to 3 ask you to read or write.

### The notebook

You are in a **notebook**: a document of **cells**, each holding prose or code.
Run a code cell with **Shift+Enter**; output appears underneath.

- **Kernel** — the Python process behind the notebook. It remembers everything
  you have defined. Cells share one, so **order matters**: skip a cell and the
  ones after it may fail on a name that was never created.
- **Restart** — *Runtime → Restart session* empties it. When a notebook
  behaves impossibly, restart and run from the top.

When a cell errors, read the **last** line of the message first. That is the
actual complaint; everything above it is the route Python took to get there.

### Step 0a: variables, types, and f-strings

A **variable** is a name bound to a value with a single `=`. Read `x = 5` as
"let x refer to 5", not "x equals 5". Every value has a **type**: `str` is
text, in quotes; `float` is a number with a decimal point; `int` is a whole
number.

An **f-string** is a string with `f` before the opening quote. Anything inside
`{ }` is evaluated and dropped into the text, and `\n` inside any string starts
a new line. The part after a colon inside the braces is a **format spec**. It
controls how a number is *displayed* and never changes the value:

| Spec | Meaning | Example |
|---|---|---|
| `:.2f` | fixed point, 2 decimals | `7.1` → `7.10` |
| `:.1%` | percent, 1 decimal | `0.042` → `4.2%` |
| `:,` | thousands separator | `2565` → `2,565` |
| `:+.3f` | always show the sign | `0.081` → `+0.081` |
| `:<14` / `:>9` | pad to 14 / 9 characters, left / right aligned | lines up columns |

Specs combine: `:>9.3f` is right-aligned in 9 characters with 3 decimals.

One trap: `:.1%` multiplies by 100 for you. Multiply by 100 yourself as well and
the number shows 100 times too big. The last two lines of the cell show it.

In [ ]:
# GIVEN — run as-is
# Step 0a: variables, types, and f-strings

country = "Switzerland"
big_mac = 7.10        # Switzerland's price, in Swiss francs
n_periods = 45
print(type(country), type(big_mac), type(n_periods))

valuation = 0.4176
print(f"{country}: CHF {big_mac:.2f}, valued {valuation:.1%} against the US")

print(f"WRONG: {valuation * 100:.1%}")
print(f"RIGHT: {valuation:.1%}")

### Step 0b: the DataFrame, one bracket vs two

`df`, loaded in Setup, is a **DataFrame**: a table with named columns and
ordered rows. The **index** is the row labels down the left; it is not a
column. A **dtype** is the type of a whole column, and many pandas errors turn
out to be a dtype that is not what you assumed.

One pair of brackets around one name, `df["name"]`, gives a **Series**: a
single column. Two pairs, `df[["name"]]`, give a DataFrame, because the inner
brackets are a *list* of column names.

In [ ]:
# GIVEN — run as-is
# Step 0b: the DataFrame, one bracket vs two

print(f"Shape (rows, columns): {df.shape}")
print(f"Index: {df.index.min()} .. {df.index.max()}")
print(f"\nDtypes:\n{df[['name', 'date', 'local_price', 'dollar_ex']].dtypes}")

# __name__ is the type's short name, without the <class ...> around it
print(f"\ndf['name']          is a {type(df['name']).__name__}")
print(f"df[['name']]        is a {type(df[['name']]).__name__}")
print(f"df[['name','date']] is a {type(df[['name', 'date']]).__name__}")

### Step 0b, continued: boolean masks

A **boolean mask** is the most important pattern in this course. A comparison
on a column gives one `True` or `False` per row. Put that inside `df[ ... ]`
and you keep the rows where it is `True`. Because `True` counts as 1, `.sum()`
of a mask counts the rows that pass.

Combine conditions with `&` (and), `|` (or) and `~` (not). The words `and`,
`or` and `not` raise *ValueError: The truth value of a Series is ambiguous*.
Each condition needs its own parentheses, because `&` binds tighter than `>`:

```python
df[(df["a"] > 1) & (df["b"] < 2)]     # right
df[ df["a"] > 1  &  df["b"] < 2 ]     # wrong
```

In [ ]:
# GIVEN — run as-is
# Step 0b: boolean masks

# pandas reads the text as a date for the comparison
is_2024 = df["date"] == "2024-07-01"
print(f"The mask is {len(is_2024)} True/False values; {is_2024.sum()} are True.")

expensive_2024 = df[is_2024 & (df["dollar_price"] > 6)]
print(f"\nBig Macs over $6 in July 2024: {len(expensive_2024)}")
# index=False prints the table without the row labels
print(expensive_2024[["name", "dollar_price"]].to_string(index=False))

### Step 0c: lists, loops, and the shape of a panel

Part 3 asks whether a table is cross-sectional, a time series or a panel, and
whether that panel is *balanced*. Both are counting questions, and counting
needs a **list** and a **`for` loop** to walk one.

A **list** is an ordered box of values in square brackets. Python counts from
0, and `x[-1]` counts from the end. The slice `x[1:3]` includes position 1 and
*excludes* position 3.

A **`for` loop** runs its indented block once per element. The indentation is
how Python knows where the block ends.

In [ ]:
# GIVEN — run as-is
# Step 0c: lists and the for loop

key_columns = ["name", "date", "local_price", "dollar_ex", "dollar_price"]
print("How many:      ", len(key_columns))
print("First (idx 0): ", key_columns[0])
print("Last  (idx -1):", key_columns[-1])
print("Slice [1:3]:   ", key_columns[1:3])

print("\n--- dtype of each key column ---")
for col in key_columns:
    print(f"  {col:<14} {df[col].dtype}")

### Step 0c, continued: counting units, periods, and groups

`.nunique()` counts the distinct values in a column. Count the units and the
periods, and you know whether you hold a cross-section, a time series or a
panel.

`.groupby(col).size()` splits the table into one group per distinct value of
`col` and counts the rows in each: split, apply, combine. Here it counts the
periods each country appears in, which is a **panel balance** check. The result
is a Series indexed by country, so a boolean mask filters it the same way it
filters a column.

In [ ]:
# GIVEN — run as-is
# Step 0c: counting units, periods, and groups

n_countries = df["name"].nunique()
n_periods = df["date"].nunique()
print(f"Units (countries): {n_countries}")
print(f"Periods (dates):   {n_periods}")
print(f"Rows if every country appeared in every period: {n_countries * n_periods:,}")
print(f"Rows actually present:                          {len(df):,}")

periods_per_country = df.groupby("name").size()
print(f"\nperiods_per_country is a {type(periods_per_country).__name__}, indexed by country:")
print(periods_per_country.head(3))

complete = periods_per_country[periods_per_country == n_periods]
print(f"\nCountries present in all {n_periods} periods: {len(complete)} of {n_countries}")
print(f"Balanced panel: {len(complete) == n_countries}")

### Step 0d: dictionaries

Part 3 asks you to write a function that returns a **dictionary**, so
dictionaries and functions are the last two pieces of this lab's slice.

A dictionary is a lookup table: you get a value back by its **key**, not by its
position. It is written with `{ }` as `key: value` pairs. A list answers "what
is third?"; a dictionary answers "what is the value for `"rows"`?". You add a
key by assigning to it, which is how Part 3's profile gets built one piece at a
time.

In [ ]:
# GIVEN — run as-is
# Step 0d: dictionaries

profile = {
    "rows": len(df),
    "columns": df.shape[1],
    "countries": df["name"].nunique(),
}
print("The dict:      ", profile)
print("One value:     ", profile["countries"])
print("Its keys:      ", list(profile.keys()))

profile["periods"] = df["date"].nunique()
profile["balanced"] = len(df) == profile["countries"] * profile["periods"]
print("After adding two more:", profile)

### Step 0d, continued: measuring what is missing

`.isna()` gives `True` or `False` for every cell: a mask over the whole table.
Because `True` counts as 1, `.mean()` of a mask is the **share** that is
`True`. On a DataFrame, `.mean()` works column by column, so `df.isna().mean()`
is the share missing in each column, and a second `.mean()` averages those.

The two loops at the end build the same numbers as a dictionary, then collect
the columns more than 10% empty into a list with `.append()`. Read the first
loop closely: Part 3 asks you to write it.

In [ ]:
# GIVEN — run as-is
# Step 0d: measuring what is missing

print(f"Share of all cells that are missing: {df.isna().mean().mean():.1%}")

missing_pct = (df.isna().mean() * 100).round(1)
print("\nMissing % by column (worst five):")
print(missing_pct.sort_values(ascending=False).head(5).to_string())

missing_by_col = {}
for col in df.columns:
    missing_by_col[col] = round(df[col].isna().mean() * 100, 1)

gappy = []
for col in df.columns:
    if missing_by_col[col] > 10:
        gappy.append(col)
print(f"\nColumns more than 10% empty ({len(gappy)}): {gappy}")

### Step 0e: `def`, packaging the work so it can be reused

A **function** is a named recipe. `def` starts it, the indented block is the
body, and `return` hands a value back to whoever called it. The names in the
parentheses are **parameters**, filled in when you call it. Writing
`unit_col="name"` gives a parameter a **default**, so the caller may leave it
out. The line in triple quotes under `def` is the **docstring**: a sentence
saying what the function does.

A function that returns a dictionary is the pattern Part 3 asks for: gather
several facts, name each one, and hand them back in one object. Because it
takes the DataFrame as an argument, the same function works on any slice of
it, which is the point of writing a function instead of a cell.

In [ ]:
# GIVEN — run as-is
# Step 0e: def

def count_units(data, unit_col="name"):
    """Count the distinct values in column `unit_col` of `data`."""
    return data[unit_col].nunique()


print("Distinct countries:", count_units(df))
print("Distinct currencies:", count_units(df, unit_col="currency_code"))


def quick_profile(data, unit_col="name", time_col="date"):
    """Return a small structural profile of `data` as a dict."""
    out = {}
    out["shape"] = data.shape
    out["n_units"] = data[unit_col].nunique()
    out["n_periods"] = data[time_col].nunique()
    out["is_balanced"] = len(data) == out["n_units"] * out["n_periods"]
    return out


result = quick_profile(df)
print("\nquick_profile(df) returns a dict:")
for key in result:            # a loop over a dict walks its keys
    print(f"  {key:<13} {result[key]}")

print("\nSame function, one cross-section:")
print(" ", quick_profile(df[is_2024]))

### Step 0f: build it — the correct PPP valuation

This is the calculation Part 1 will hand you with an error in it. Build the
right answer now, and keep the output on screen to compare against.

**The economics.** Implied PPP is the exchange rate that *would* make a Big Mac
cost the same in both countries. Valuation asks how far the *actual* market
rate sits from it:

    implied_ppp   = local price / US price
    valuation_pct = (implied_ppp - actual rate) / actual rate * 100

Read the sign: implied PPP *above* the actual rate means the local currency
buys less than PPP says it should, so it is **overvalued**. Getting this ratio
upside down is a real and common error, and it silently reverses every
conclusion.

Three new pieces of pandas: `.copy()` makes a separate table, so adding columns
to it leaves `df` alone; `.loc[rows, column]` picks rows by a mask and one
column by name; and `.nlargest(5, col)` keeps the five rows with the highest
values of `col`.

In [ ]:
# GIVEN — run as-is
# Step 0f: the correct PPP valuation

cs2024 = df[is_2024].copy()
# .values[0] takes the single number out of the one-row result
us_price = cs2024.loc[cs2024["iso_a3"] == "USA", "dollar_price"].values[0]
print(f"US benchmark price: ${us_price:.2f}\n")

cs2024["implied_ppp"] = cs2024["local_price"] / us_price
cs2024["valuation_pct"] = (
    (cs2024["implied_ppp"] - cs2024["dollar_ex"]) / cs2024["dollar_ex"] * 100
)

show = ["name", "valuation_pct"]
print("Most OVERVALUED (correct):")
print(cs2024.nlargest(5, "valuation_pct")[show].to_string(index=False))
print("\nMost UNDERVALUED (correct):")
print(cs2024.nsmallest(5, "valuation_pct")[show].to_string(index=False))
print(f"\nMedian valuation: {cs2024['valuation_pct'].median():.1f}%")

### Now diagnose

Switzerland should head the overvalued list, with Taiwan and Indonesia at the
far end of the undervalued one. That is your baseline — **keep this output visible**.

Part 1 hands you the same calculation with one thing changed. Do not read the
code looking for a typo. Run it, look at the *answer*, and ask whether it can
be true. A currency that a moment ago was 60% cheap does not become the most
expensive in the world because of a rounding error.

---

## Part 1: Find the Bug — PPP Computation (10 min)

The following code computes Big Mac PPP valuations.
**Something is wrong with the formula.** Find it, fix it, explain.

In [ ]:
# GIVEN — run as-is (it contains the error Part 1 asks you to find)
# Part 1: PPP computation — find the bug

df_2024 = df[is_2024].copy()
us_price = df_2024.loc[df_2024["iso_a3"] == "USA", "dollar_price"].values[0]

df_2024["implied_ppp"] = df_2024["local_price"] / us_price
df_2024["valuation_pct"] = (
    (df_2024["dollar_ex"] - df_2024["implied_ppp"]) / df_2024["implied_ppp"] * 100
)

top5 = df_2024.nlargest(5, "valuation_pct")[["name", "valuation_pct"]]
print("Top 5 'overvalued' currencies:")
print(top5.to_string(index=False))
print()
# .iloc[0] is the first row by position, .iloc[1] the second
print(f"Conclusion: {top5['name'].iloc[0]} and {top5['name'].iloc[1]} are the most "
      f"OVERVALUED currencies in the world!")

### YOUR DIAGNOSIS

1. **What is wrong?** (identify the specific line and the mathematical error)
2. **Why does the bug produce backwards results?** (Indonesia and Egypt should be undervalued, not overvalued)
3. **Fix the code below** and report the correct top 5 overvalued countries

**Verification checkpoint:** After fixing, the July 2024 top five overvalued
should read Switzerland **+41.8%**, Uruguay +24.3%, Norway +18.9%,
Argentina +15.0%, Euro area +6.5%. Switzerland must come first; anything in
the +35% to +50% range is the right answer for it (the file this cell
downloads is The Economist's live repository, so the last decimal moves when
they revise a back-series). The five you saw before the fix — Taiwan,
Indonesia, Egypt, India, South Africa — should now be the five most
*under*valued, at about -60% to -50%. If Indonesia is still in the top 5 overvalued,
you haven't found the bug.

In [ ]:
# YOUR TASK — Fix the PPP computation
# Overwrite df_2024["valuation_pct"] with the corrected formula, then print the new top five.

### Document your diagnosis

*Your answers here:*

1. **What is wrong** —
2. **Why it runs backwards** —


### Now look at it

A ranking is hard to read as a column of numbers and easy to read as a chart.
The chart reads `df_2024` *after* your fix, so it also double-checks the fix:
Switzerland belongs at the top. If Taiwan and Indonesia still lead, the bug is
still there; go back to the fix cell.

The bars are the eight lowest and the eight highest valuations. The United
States is the benchmark, so its bar is exactly 0 by construction; that is not a
missing value. The chart is also saved to `figures/ch01_valuation.png`.

In [ ]:
# GIVEN — run as-is, after your fix
# Part 1: see the corrected ranking

Path("figures").mkdir(exist_ok=True)     # Colab starts with no figures/ folder

ranked = df_2024.sort_values("valuation_pct")
ends = pd.concat([ranked.head(8), ranked.tail(8)])    # stack the two tables

# np.where(mask, a, b) gives a where the mask is True and b where it is False
colors = np.where(ends["valuation_pct"] < 0, "firebrick", "navy")

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(ends["name"], ends["valuation_pct"], color=colors)
ax.axvline(0, color="black")
ax.set_xlabel("Valuation against the US (%)")
ax.set_title("Big Mac Index, July 2024 — eight most under- and overvalued")
plt.tight_layout()
plt.savefig("figures/ch01_valuation.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Median valuation: {df_2024['valuation_pct'].median():.1f}%")
print(f"Dearer than the US: {(df_2024['valuation_pct'] > 0).sum()} of {len(df_2024)}")
print("Saved to figures/ch01_valuation.png")

### The economics behind the ranking

The bug was arithmetic. These three are not — they ask what the corrected
chart actually means. Question 3 is the one worth most of your time.

1. **Why are most countries undervalued against the US?** What does that tell
   you about the US price level — and about the Big Mac as a *measuring
   instrument*?
2. **Switzerland has been the most overvalued in every period since 2015.** Which non-tradeable
   inputs explain it? (rent, wages, protection, tax)
3. **Critical.** If a hedge fund used this chart to place currency bets, would
   you expect it to make money? Why or why not?

*Your answers here:*

1.
2.
3.


## Part 2: Find the Methodological Flaw — Missing Data Handling (10 min)

The following analysis handles missing data before computing summary statistics.
The code runs correctly. The methodology is wrong. Find the flaw.

In [ ]:
# GIVEN — run as-is (its method is what Part 2 asks you to diagnose)
# Part 2: missing data handling

max_periods = df["date"].nunique()
country_counts = df.groupby("name")["date"].count()
incomplete = country_counts[country_counts < max_periods]

print(f"Countries with incomplete panels: {len(incomplete)}")
print(incomplete.sort_values())
print()

# Keep only the countries present in every period.
# .index holds a Series' labels, here country names; .isin() tests membership.
complete_countries = country_counts[country_counts == max_periods].index
df_clean = df[df["name"].isin(complete_countries)].copy()

print(f"Dropped {df['name'].nunique() - df_clean['name'].nunique()} countries")
print(f"Remaining: {df_clean['name'].nunique()} countries with complete panels")
print()

avg_price = df_clean.groupby("date")["dollar_price"].mean()
print("Average Big Mac price (complete-panel countries only):")
print(avg_price.tail())
print()
print("CONCLUSION: The global average Big Mac price has risen steadily.")
print("This represents the true trend for ALL countries worldwide.")

### YOUR DIAGNOSIS

Five questions. Measure the answer to 4 in the next cell, then write all five
in the *Document your diagnosis* cell after it.
Three and five are the ones this course is really asking about.

1. **What is the methodological flaw?** The code runs and the arithmetic is
   right. The reasoning is wrong.
2. **Which countries were dropped — are they a random subset?** Read the
   printed list before you characterise it.
3. **Which missing data mechanism is at work?** MCAR, MAR, or MNAR — and
   check whether one label covers every gap in this panel.
4. **Which direction is the "global average" biased, and by how much?**
   Measure it in the next cell rather than arguing about it.
5. **Propose a better approach** that acknowledges the missing data instead of
   silently dropping it.

**Checkpoint — 32 of 57 dropped, 25 kept.** Before writing "emerging markets get
dropped", check three names in the output above: **Argentina** is *kept* (all 45
periods), while **Norway** (41) and **Denmark** (44) are *dropped*. The filter
selects on continuity of measurement, not on development — and it conflates
three different things. Some countries **left** the index (Russia, last priced
January 2022; Sri Lanka, January 2024). Some **joined late** (the Gulf and
Central American block, 17 periods each from July 2018). And some have **gaps
in the middle** of a run that is still going (Venezuela 31, Ukraine 39 — both
priced in July 2026). Check first and last dates before you assign a country
to a group. Two traps in the names: `UAE` (to January 2018) and `United Arab
Emirates` (from July 2018) are one country renamed, and `Euro area` is not a
country — so the 57 names are 56 ISO codes and 55 economies. That split
matters for question 3.

In [ ]:
# YOUR TASK — Demonstrate the bias
# Two blanks. Both use variables the cell above already made.

# (a) the flawed approach: complete-panel countries only       [Step 0c]
#     `df_clean` above is already filtered to those 25.
complete_only = ____

# (b) the honest one: every country available in each period   [Step 0c]
all_available = ____

# The gap between the two is the bias. Nothing below needs editing.
gap = complete_only - all_available
print(f"mean overstatement: ${gap.mean():+.3f}   ({(gap / all_available).mean():+.1%})")
print(f"complete-panel average is higher in {(gap > 0).sum()} of {len(gap)} periods")

# A dict of Series becomes a table: one column per key, rows lined up by date
table = pd.DataFrame({"complete": complete_only, "all avail.": all_available, "gap": gap})
print("\nlast six periods:")
print(table.tail(6).round(3))

In [ ]:
# CHECK — run as-is after the two blanks
# assert stops the cell with its message when the condition is False.

def check_mean_by_date(series, frame, name):
    """Assert that `series` is the average dollar price of `frame` on each date."""
    count = frame.groupby("date").size()
    total = frame.groupby("date")["dollar_price"].sum()
    # isinstance(x, pd.Series) is True when x is a Series
    assert isinstance(series, pd.Series), f"{name} should be a Series indexed by date."
    # .equals() is True when both have the same dates in the same order
    assert series.index.equals(total.index), f"{name} is not indexed by the dates of its frame."
    # a mean times its count is the total; np.allclose compares with room for rounding
    assert np.allclose(series * count, total), (
        f"{name} is not the average dollar price, per date, of the frame it should use.")


check_mean_by_date(complete_only, df_clean, "complete_only")
check_mean_by_date(all_available, df, "all_available")
print("Part 2 check passed.")

**Measured** (The Economist's live file; the last decimal moves when they
revise): mean overstatement **+\$0.081 (+2.1%)**, and the complete-panel
average is higher in **33 of 45** periods.

### Document your diagnosis

*Your answers to the five questions here:*

1. **The flaw** —
2. **Who was dropped, and is it random** —
3. **Which mechanism (MCAR / MAR / MNAR)** —
4. **Direction and size of the bias** —
5. **A better approach** —


## Part 3: Data Structure Profiling (GUIDED + YOUR TASK — 14 min)

Part 2 was about rows that are missing. This part is about what the *shape* of
the data tells you before you model anything. The given cell first, then you
package it into a function.

Step 0c counted 57 units and 45 periods. Those two counts decide which of the
three structures you are holding, and the cell below gets all three out of
this one frame, depending on how it is sliced. Its last three lines count the
rows this panel lacks. Those rows are the subject of Part 2, and they are
*absent*, not null: no `.isna()` check anywhere will find them.

In [ ]:
# GIVEN — run as-is
# Part 3: the same data, sliced three ways

# Cross-sectional: many units, one period
cross = df[is_2024]
print(f"Cross-sectional : {cross['name'].nunique():>3} countries x "
      f"{cross['date'].nunique()} period   -> {len(cross):>4} rows")

# Time series: one unit, many periods
one_country = df[df["name"] == "Switzerland"]
print(f"Time series     : {one_country['name'].nunique():>3} country   x "
      f"{one_country['date'].nunique()} periods -> {len(one_country):>4} rows")

# Panel: many units, many periods
print(f"Panel (all)     : {n_countries:>3} countries x "
      f"{n_periods} periods -> {len(df):>4} rows")

full = n_countries * n_periods
print(f"\nIf every country appeared in every period: {full:,} rows")
print(f"Actually present:                          {len(df):,} rows")
print(f"Absent:                                    {full - len(df):,} "
      f"({1 - len(df) / full:.1%} of the grid)")

In [ ]:
# YOUR TASK — Profile the structure of any DataFrame
# Five blanks, one line each. Replace every ____

def profile_dataframe(data, unit_col="name", time_col="date"):
    """Return a dict describing the structure and completeness of `data`."""
    profile = {}
    profile["shape"] = data.shape

    # 1. How many units, and how many periods?                    [Step 0c]
    profile["n_units"]   = ____
    profile["n_periods"] = ____

    # The structure follows from those two counts alone
    if profile["n_units"] > 1 and profile["n_periods"] > 1:
        profile["structure"] = "panel"
    elif profile["n_periods"] > 1:
        profile["structure"] = "time series"
    else:
        profile["structure"] = "cross-sectional"

    # 2. Count the periods each unit actually appears in.         [Step 0c]
    periods_per_unit = ____

    # 3. Keep only the units that appear in every period, and say whether
    #    that is all of them.                                     [Step 0c]
    complete = periods_per_unit[periods_per_unit == profile["n_periods"]]
    profile["complete_units"] = len(complete)
    profile["balanced"] = ____

    # 4. The share of each column that is missing, as a percentage.
    #    One entry per column, built with the loop.               [Step 0d]
    missing = {}
    for col in data.columns:
        missing[col] = ____
    profile["missing"] = missing

    return profile


# Run it on the Big Mac panel. The missing dict is long, so it is summed up
# as one count at the end.
result = profile_dataframe(df)
for key in result:
    if key != "missing":
        print(f"  {key:<16} {result[key]}")

n_gappy = 0
for col in result["missing"]:
    if result["missing"][col] > 10:
        n_gappy += 1          # += 1 adds one to the count
print(f"  cols >10% missing {n_gappy}")

In [ ]:
# CHECK — run as-is after the blanks
# Toy frames small enough to check by eye, so the check does not give away the Big Mac answer.
toy = pd.DataFrame({"name": ["A", "A", "B", "B", "C"],
                    "date": ["t1", "t2", "t1", "t2", "t1"],
                    "x":    [1.0, None, 3.0, 4.0, 5.0]})
p = profile_dataframe(toy)
assert (p["n_units"], p["n_periods"]) == (3, 2), (
    f"toy has 3 units and 2 periods; got {p['n_units']} and {p['n_periods']}.")
assert p["complete_units"] == 2, f"toy has 2 complete units; got {p['complete_units']}."
assert isinstance(p["balanced"], (bool, np.bool_)), "balanced should be True or False."
assert not p["balanced"], "toy is unbalanced (C is missing t2), but balanced is True."
assert np.isclose(p["missing"]["x"], 20.0), (
    f"x is missing 1 value in 5 = 20 percent; got {p['missing']['x']}.")

square = toy[toy["name"] != "C"]
assert profile_dataframe(square)["balanced"], "A and B both have t1 and t2, so it is balanced."
print("Part 3 check passed.")

**Measured on the Big Mac panel:** 57 units, 45 periods, panel, 25 complete,
balanced False, 7 columns more than 10% missing.

---

### Next: Lab 2 turns this into a module

Keep the `profile_dataframe()` you wrote in Part 3. Lab 2's Part 0 covers
`raise` and the two ways a notebook can write a real `.py` file, and its Part 4
packages this function into one.

---
## Digital Portfolio: your README

### Generate your README
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own Part 2 output before you paste it.

In [ ]:
# README prompt — documentation only, no code

# PASTE THIS PROMPT INTO CLAUDE:
#
# "I need help writing a project description for my data science lab.
# **Important Rule:** Do NOT generate any Python code for me.
#
# **What I did in this lab:**
# * Diagnosed and fixed a PPP computation bug (swapped numerator/denominator)
# * Showed that dropping every country with an incomplete panel biases
#   the 'global' average: the complete-panel average Big Mac price is
#   $[YOUR VALUE] ([YOUR VALUE]%) above the all-available average, and
#   higher in [YOUR VALUE] of 45 periods
# * Wrote profile_dataframe(), which reports units, periods, structure,
#   complete units, balance and per-column missing percentages
#
# **Please write a README.md entry including:**
# 1. Project Title: Data Quality Profiling — Big Mac Index
# 2. Objective: one sentence
# 3. Methodology: Bullet points of technical steps
# 4. Key Findings: Summary of results
# Write it plainly, in the first person. No words like 'robust',
# 'comprehensive' or 'leverage'. Use only the numbers I gave you."

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab01-data-portfolio`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab01-data-portfolio`, tick **Add a README file**, and create it.
2. **Run the whole notebook top to bottom** (**Runtime → Run all**) and check your
   output is showing. The rendered output — charts included — is what is graded.
3. In Colab: **File → Save a copy in GitHub**.
   - **First time only:** a GitHub window asks you to authorize Colab. Sign in, then
     authorize. **If your repo is Private, tick the box granting access to private
     repositories first** — otherwise Colab won't list your repo.
   - Choose `econ5200-lab01-data-portfolio`, branch `main`, commit message
     `Lab 01 submission`, tick **Include a link to Colab**, and click **OK**.
4. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your **repository** URL in the **Comments**
   box — `https://github.com/yourname/econ5200-lab01-data-portfolio`, not the
   link to the notebook file. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install. Stuck? The GitHub Setup and Repository Guide page has a
Troubleshooting section.
